[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BMGLab/BFB/blob/main/W02_Files__coordinates_and_reference_versions.ipynb)

# Week 02 | Files, coordinates and reference versions

**Core practical: 45 minutes.** Validate interval lengths and identify missing reference metadata.

No paid AI tool, local installation or external dataset download is required. Open it in Colab with the badge above, then **File > Save a copy in Drive** before you start so your work is kept. Run the cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

## Before running (5 min)
A genome is a sequence; an annotation adds named features on a particular sequence version.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. A seed supports reproducibility; it is not proof of authorship.

In [ ]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "demo-001"  # Replace with your assigned course pseudonym, not your name or national ID.
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

### Who is submitting (1 min)
Fill in your **full name** and your **Ege student number**, then run the cell. It refuses to
continue if either is missing or malformed, so a mistyped digit is caught here — in the room,
where it takes ten seconds to fix — rather than after the deadline.

Your `COURSE_ID` above still deals your dataset. This is only about attributing the work to you.

In [ ]:
import os
if not os.path.exists("bib_colab.py"):
    !curl -sfO https://raw.githubusercontent.com/BMGLab/BFB/main/bib_colab.py
import bib_colab as bib

A = bib.start("W02")
A.whoami(
    name="",          # your full name, e.g. "Ayşe Gül Öztürk"
    student_no="",    # your Ege student number, digits only
    section="EN",     # "EN" or "TR"
)

### Prediction
Write your prediction here before running the investigation, then copy it into PREDICTION in the response cell.

## Guided investigation (20 min)
1. Read each record and name its purpose.
2. Convert the BED interval to one-based inclusive coordinates.
3. Compare extracted bases and lengths.
4. Find the defect in the intentionally incorrect conversion.

In [ ]:
seq = "ACGTACGTAA"
bed = {"chrom": "toy_contig", "start": 2, "end": 7}
gtf = 'toy_contig\tclass\texon\t3\t7\t.\t+\t.\tgene_id "toy_G1";'
vcf = 'toy_contig\t4\t.\tT\tC\t.\t.\t.'
# Toy contig, not a human assembly. This is a teaching record, not a clinical variant.
start_1, end_1 = bed["start"] + 1, bed["end"]
selected = seq[bed["start"]:bed["end"]]
assert len(selected) == bed["end"] - bed["start"] == end_1 - start_1 + 1
wrong_start, wrong_end = bed["start"] + 1, bed["end"] + 1
print("BED", bed, "inclusive", (start_1, end_1), "bases", selected)
print("Deliberately wrong inclusive interval:", (wrong_start, wrong_end))
print("GTF:", gtf, "\nVCF:", vcf)
RESULTS = {"data_status": "SYNTHETIC", "selected": selected, "length": len(selected),
           "reference_build": "toy_contig; no human assembly", "inclusive": [start_1, end_1]}

## Explain the evidence (10 min)
**Q1.** Explain why adding one to both BED endpoints changes the region.

**Q2.** Are BED [0,5) and [5,8) overlapping? Explain by enumerating bases.

**Q3.** What can you conclude about a reference build from a header containing only chr1?

In [ ]:
PREDICTION = ""  # Fill before the analysis.
RESPONSES = {"Q1": "", "Q2": "", "Q3": ""}
AI_DISCLOSURE = "No AI used."  # Change to tool, date, purpose, and checks if you used one.
CHECK_PERFORMED = ""  # Describe one actual check, even if it found no error.

## Save and submit (5 min)
Complete your responses above, then **Runtime > Restart session and run all** so every number in
the notebook is the one your answers describe.

Then run the cell below. It checks that nothing is missing, prints a receipt code, and sends this
notebook straight to your instructor. There is nothing to download and nothing to upload.

A completion check looks for the presence of your responses, not for scientific correctness. If
the upload fails, the cell prints your receipt code and what to do instead — follow it before you
leave. Paper fallback may be submitted as a legible scan with the same answers.

In [ ]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":2, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W02_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

# --- submit -------------------------------------------------------------
A.answers(RESPONSES,
          prediction=PREDICTION,
          check=CHECK_PERFORMED,
          disclosure=AI_DISCLOSURE,
          results=RESULTS)
A.check()
A.submit()

## Paper / device-free route
Draw ten boxes labelled 1-10 above and 0-9 below. Shade positions 3-7. Check five bases GTACG.

## Optional extension
Optional: inspect how a VCF deletion includes an anchor base; no indel normalization is examined.

## Sources
- [S03] SAM/BAM, VCF and BED specifications, maintained by the HTS community. https://samtools.github.io/hts-specs/
- [S04] UCSC Genome Browser: data format FAQ. https://genome.ucsc.edu/FAQ/FAQformat.html
- [S05] Python 3 tutorial: introduction and control flow. https://docs.python.org/3/tutorial/